In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys, os
from pathlib import Path

project_root = Path(os.getcwd()).parent if 'spaghetti_code' in os.getcwd() else Path(os.getcwd())
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cs_priors.plotting.plot_complex import plot_matrices
from cs_priors.simulation.mixing_model import quick_sector_sim, quick_sim
from cs_priors.solvers.group_lasso import group_lasso_solve
from cs_priors.plotting.plot_signals import plot_recovery
from cs_priors.plotting.plot_geometry import plot_geometry_on_ax, plot_sim_geometry
from cs_priors.metrics.count_sparsity import (
    noise_threshold,
    active_sources,
    source_energies,
    detection_scores,
    relative_error,
    score,
)
from cs_priors.metrics.benchmark import run_methods, score_methods, grid_benchmark, heatmap_benchmark
from cs_priors.solvers.vectorized_sparse_prior import sparse_prior_solution
from cs_priors.solvers.group_lasso import _to_block_system

def sparse_prior_solve(sim):
    A_block, Y_block, S, N = _to_block_system(sim)
    X_flat = sparse_prior_solution(Y_block, A_block)
    return X_flat.reshape(S, N)

from cs_priors.solvers.complex_lasso import complex_lasso

def complex_lasso_solve(sim, alpha=1e-4):
    A_block, Y_block, S, N = _to_block_system(sim)
    X_flat = complex_lasso(Y_block, A_block, alpha=alpha)
    return X_flat.reshape(S, N)

### Agenda
1. Trunkering av frekvenser ser ikke ut til å ha hjulpet.
2. P(x_opt) > P(x_opt + Bz)
3. Initialisere med støy pseudoinvers.


In [ ]:
# sjekke at optimum er globalt
sim = quick_sector_sim(num_mics=6, num_sources=10, num_active=2, sampling_rate=16000, duration=0.004, source_distance=0.2, mic_radius=0.05, mode="sine", snr_db=None, min_freq_hz=200)
plot_sim_geometry(sim)

### Simuleringsparametere

In [ ]:
num_sources = 10
num_seeds = 0
simulation_type = [quick_sector_sim, quick_sim][0]

# These are all keyword arguments
param_grid = {
    "num_mics":    [6,7,8,9,10],
    "num_sources": [10],
    "num_active":  [2],
    "sampling_rate": [16000], # 16000 = 16 kHz
    "duration": [0.004], # 0.002 = 2 ms 
    "source_distance": [0.2], # 0.01 = 10 cm
    "mic_radius": [0.05], # 0.01 = 1 cm
    "mode": ["sine"],
    "snr_db": [None],
    "min_freq_hz": [200], # 1000 = 1 kHz
}

methods = {
    "sparse_prior_no_group": sparse_prior_solve,
    "complex_lasso": complex_lasso_solve,
}

# Visualize the simulation we will be working with
last_params = {k: v[0] for k, v in param_grid.items()}
sim = simulation_type(**last_params, seed=0)

plot_geometry_on_ax(ax=plt.gca(), mics=sim.mics, sources=sim.sources, pad_factor=0.12)
plt.title("Example Simulation Geometry")
plt.gca().figure.dpi = 50
plt.show()

# check the maximum is the maximum


X_sp = sparse_prior_solve(sim)
X_rlasso = complex_lasso_solve(sim, alpha=1e-4)

# print frequencies
print("Frequencies included in the simulation:")
print(sim.freqs)


plot_matrices([sim.X, sim.X_pinv, X_sp, X_rlasso], titles=["True Sources", "Pseudoinverse", "Sparse Prior", "Complex Lasso"], show_values=False, dpi=50)



# Without noise

In [ ]:
metric="precision"
pivot_y = "num_mics"
pivot_x = "num_active"
# param_grid["snr_db"] = [None]

heatmap_benchmark(sim_factory=simulation_type, param_grid=param_grid, methods=methods, num_seeds=num_seeds, metric=metric, pivot_x=pivot_x, pivot_y=pivot_y)

# With Noise

In [ ]:
param_grid["snr_db"] = [1000]
pivot_x = "snr_db"

heatmap_benchmark(sim_factory=simulation_type, param_grid=param_grid, methods=methods, num_seeds=num_seeds, metric=metric, pivot_x=pivot_x, pivot_y=pivot_y)